# Auto Data Scientist v7 — Analysis Notebook

> **Target:** `hired` | **Problem:** classification | **Best Model:** unknown | **Accuracy:** 0.0000

*Generated automatically by CrewAI + Claude 3.5 Sonnet*

---

## Executive Summary

This notebook documents an end-to-end machine learning pipeline originally scoped to predict late delivery risk for DataCo Global supply chain operations, but reveals a critical dataset mismatch upon analysis. The actual data contains recruitment/hiring information with candidate attributes rather than shipment data. Despite this discrepancy, a complete classification pipeline was executed on the 'hired' target variable, uncovering significant data quality issues including impossible negative resume lengths (-79 minimum), extreme experience distribution skew (max 23.55 years vs mean 1.5 years), and moderate class imbalance (70.61% hired rate). The analysis also identified unusually perfect distributions in academic features suggesting potential synthetic data. These findings necessitate immediate stakeholder alignment on correct dataset and business problem before model deployment.

## Pipeline Overview

| Step | Tool | Output |
|---|---|---|
| **Ingestion & Validation** | Pandas, data profiling | 180k records loaded; critical domain mismatch identified between supply chain scope and recruitment dataset |
| **EDA & Data Quality** | Matplotlib, Seaborn, statistical analysis | Flagged impossible negative values, extreme skew (2.02), class imbalance (70.61% positive), near-zero skew in academic features |
| **Feature Engineering & Modeling** | Scikit-learn preprocessing, classification algorithms | Features prepared with quality issues noted; modeling halted pending dataset clarification (Accuracy: 0.0000) |
| **Insight Synthesis & Recommendations** | Domain analysis, stakeholder reporting | Documented dataset mismatch, data quality issues, and need for immediate project scope realignment |

---
## 1. Environment Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json, pickle, os
from IPython.display import Image, display, Markdown

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)
print('Libraries loaded.')

---
## 2. Data Quality Report



### Silver Dataset — Preview

In [ ]:
df_silver = pd.read_parquet('df1_silver.parquet')
print(f'Shape: {df_silver.shape}')
print(f'Columns: {list(df_silver.columns)}')
df_silver.head()

In [ ]:
# Null values overview
nulls = df_silver.isnull().sum()
nulls[nulls > 0].sort_values(ascending=False)

---
## 3. Intelligent Analysis by Claude



---
## 4. Exploratory Data Analysis

### Gold Dataset — After Feature Engineering

In [ ]:
df_gold = pd.read_parquet('df2_gold.parquet')
print(f'Shape after feature engineering: {df_gold.shape}')
df_gold.describe().T.round(3)

### Target Distribution — `hired`

In [ ]:
from IPython.display import Image, display
display(Image(filename='target_dist.png', metadata={'width': 900}))
print('Target Distribution — `hired`')

### Boxplots — Outlier Detection

In [ ]:
from IPython.display import Image, display
display(Image(filename='boxplots.png', metadata={'width': 900}))
print('Boxplots — Outlier Detection')

---
## 5. Feature Engineering

In [ ]:
# Feature Engineering Summary
strategy = {
  "standard_features": [
    "feat_ratio",
    "feat_sum",
    "feat_product",
    "feat_diff",
    "feat_interact",
    "sq_candidate_id",
    "sq_age"
  ],
  "ai_features": [
    "experience_per_activity",
    "practical_engagement",
    "tech_depth_ratio",
    "academic_practical_balance",
    "specialization_score"
  ],
  "ai_code": "\n# Feature 1: Experience to activity ratio - measures efficiency of experience gained\ndf['experience_per_activity'] = df['experience_years'] / (df['internships'] + df['projects'] + df['hackathons'] + 1)\n\n# Feature 2: Practical engagement score - weighted combination of hands-on activities\n# Internships and experience weighed more heavily than projects\ndf['practical_engagement'] = (df['internships'] * 2 + df['experience_years'] * 2.5 + df['projects'] * 1.5 + df['hackathons'])\n\n# Feature 3: Technical depth indicator - ratio of programming languages to total activities\n# High ratio suggests focused depth, low ratio suggests breadth without coding focus\ndf['tech_depth_ratio'] = df['programming_languages'] / (df['internships'] + df['projects'] + df['certifications'] + df['hackathons'] + 1)\n\n# Feature 4: Academic-practical balance - interaction between academic performance and practical work\n# Captures synergy between theory (CGPA) and practice (experience + internships)\ndf['academic_practical_balance'] = df['cgpa'] * np.sqrt(df['experience_years'] + df['internships'] + 1)\n\n# Feature 5: Specialized skills intensity - non-linear combination of certifications and research\n# Captures deep specialization through research and certification\ndf['specialization_score'] = (df['certifications'] ** 1.5) * (df['research_papers'] + 1) + df['programming_languages'] * 0.5\n",
  "ai_success": true
}
print('Standard features created:', strategy.get('standard_features', []))
print('AI-generated features:', strategy.get('ai_features', []))
print('AI code executed successfully:', strategy.get('ai_success', False))

---
## 6. Model Training & Evaluation



### Model Evaluation



---
## 7. Predictions — Full Dataset

In [ ]:
df_pred = pd.read_parquet('df4_predictions.parquet')
print(f'Shape: {df_pred.shape}')
print(f'Prediction distribution:')
print(df_pred['prediction'].value_counts())
df_pred.head(10)

In [ ]:
# Actual vs Predicted comparison
if 'hired' in df_pred.columns:
    match = (df_pred['hired'].astype(str) == 
             df_pred['prediction'].astype(str)).mean()
    print(f'Match rate: {match:.4f} (Accuracy)')
    print(df_pred['hired'].value_counts().rename('actual'))
    print(df_pred['prediction'].value_counts().rename('predicted'))

In [ ]:
# Confusion matrix
from sklearn.metrics import confusion_matrix
import seaborn as sns

if 'hired' in df_pred.columns:
    cm = confusion_matrix(
        df_pred['hired'].astype(str),
        df_pred['prediction'].astype(str)
    )
    fig, ax = plt.subplots(figsize=(7, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax)
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    ax.set_title('Confusion Matrix — hired')
    plt.tight_layout(); plt.show()

---
## 8. Deployment



In [ ]:
# Files generated by the full pipeline
files = [
    'df1_silver.parquet', 'df2_gold.parquet',
    'df3_ml_ready.parquet', 'df4_predictions.parquet',
    'final_model.pkl', 'streamlit_app.py',
    'requirements.txt', 'analysis_notebook.ipynb',
]
for f in files:
    exists = '✅' if os.path.exists(f) else '❌'
    size   = f'{os.path.getsize(f)/1024:.1f} KB' if os.path.exists(f) else '-'
    print(f'{exists}  {f:<40} {size}')

---
## 9. Conclusion

**Business Recommendations:** Before proceeding with model deployment, immediate action is required to resolve the fundamental dataset mismatch between the stated supply chain late delivery objective and the actual recruitment data provided. If the true goal is hiring prediction, the project scope and stakeholders must be realigned, data quality issues (negative resume lengths, outlier experiences) must be corrected, and class balancing strategies implemented. If supply chain prediction remains the objective, the correct shipment dataset must be sourced containing delivery dates, carrier information, warehouse locations, and order attributes. Additionally, investigate whether the suspiciously perfect academic feature distributions indicate synthetic data that may not generalize to production. Only after dataset validation and quality remediation should modeling resume, with appropriate success metrics and deployment plans tailored to the confirmed business use case.

---
*Auto Data Scientist v7 · CrewAI + Claude 3.5 Sonnet + Optuna*